# 03 — Load + embed

Parse the fixtures, MERGE them into Neo4j, materialise the cross-jurisdiction `IMPLEMENTS` edge from DK lbk → EU regulation, then chunk + embed every Provision into the vector index.

After this notebook the graph is queryable end-to-end (notebook 04).

## Prereqs
- Notebook 02 ran (`apply_schema` happened; Neo4j alive).
- Vector index dim = 1024 (default). If you swapped to Qwen3-Embedding-8B (4096), set `EMBEDDING_DIM=4096` in `.env` and re-run notebook 02's rebuild step.

## What gets loaded
- EU regulation: GDPR (`eu/celex/32016R0679`) — 2 articles
- EU directive: Digital Content Directive (`eu/celex/32019L0770`) — 2 articles
- DK lbk: Databeskyttelsesloven (`dk/lbk/2018/502`) — 5 sub-statute Provisions in explode mode
- IMPLEMENTS edges seeded from the DK preamble's EU references (GDPR + LED)

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

FIX = Path('tests/clg/fixtures')
EU_REG = FIX / 'eurlex' / '32016R0679.en.fmx4.xml'
EU_DIR = FIX / 'eurlex' / '32019L0770.en.fmx4.xml'
DK_LBK = FIX / 'retsinformation' / 'lbk-2018-502.xml'
for p in (EU_REG, EU_DIR, DK_LBK):
    assert p.exists(), f'fixture missing: {p}'
print('all fixtures present')

In [ ]:
from crimellm.clg.config import get_settings
from crimellm.clg.graph import (
    apply_schema, get_store,
    load_chunks, load_implements, load_instruments, load_provisions,
)

get_settings.cache_clear()
store = get_store()
store.verify()
print('connected to', store.settings.neo4j_uri)
print('embedding model:', store.settings.embedding_model)
print('embedding dim  :', store.settings.embedding_dim)

## Step 1 — clean slate (optional)

Drop existing Instruments/Provisions/Cases/Chunks so this notebook is idempotent. Schema (constraints + indexes) survives.

In [ ]:
with store.session() as s:
    s.run('MATCH (n) WHERE n:Instrument OR n:Provision OR n:Case OR n:Chunk DETACH DELETE n')
rows = store.run('MATCH (n) RETURN labels(n) AS lbls, count(*) AS n')
print('after wipe:', rows)

## Step 2 — parse + load EU regulation + directive

`parse_regulation_file` returns `(Instrument, [Provision], cites_celex)`. The `cites_celex` list is the IMPLEMENTS-seed candidate set extracted from preamble + footnotes (Phase 3.2).

In [ ]:
from crimellm.clg.parse import eurlex as P_EU

eu_reg = P_EU.parse_regulation_file(EU_REG)
eu_dir = P_EU.parse_regulation_file(EU_DIR)

print(f'{eu_reg.instrument.id:<32} {eu_reg.instrument.short_title[:60]!r}')
print(f'  provisions: {[p.section_path for p in eu_reg.provisions]}')
print(f'  cites CELEX: {eu_reg.cites_celex}')
print()
print(f'{eu_dir.instrument.id:<32} {eu_dir.instrument.short_title[:60]!r}')
print(f'  provisions: {[p.section_path for p in eu_dir.provisions]}')
print(f'  cites CELEX: {eu_dir.cites_celex}')

load_instruments([eu_reg.instrument, eu_dir.instrument], store=store)
load_provisions(list(eu_reg.provisions) + list(eu_dir.provisions), store=store)

## Step 3 — parse + load DK lbk

`parse_statute_file` returns `(Instrument, [Provision], cites_eu_celex)` where `cites_eu_celex` is the **DK→EU CELEX seed list** for the IMPLEMENTS edge (Phase 4.2). DK preambles use surface forms like `forordning (EU) 2016/679` and `direktiv 95/46/EF`; the extractor converts them to canonical CELEX.

In [ ]:
from crimellm.clg.parse import retsinformation as P_DK

dk = P_DK.parse_statute_file(DK_LBK, doc_type='lbk', year=2018, num=502)
print(f'{dk.instrument.id:<32} {dk.instrument.short_title!r}')
print(f'  provisions: {len(dk.provisions)}')
for p in dk.provisions:
    print(f'    {p.section_path:<24} {p.id}')
print(f'  cites EU CELEX: {dk.cites_eu_celex}')

load_instruments([dk.instrument], store=store)
load_provisions(dk.provisions, store=store)

## Step 4 — materialise IMPLEMENTS edges (Phase 3.4)

Each `(dk_id, eu_celex_id, raw_ref)` triple becomes an `(Instrument)-[:IMPLEMENTS]->(Instrument)` edge. Missing-target rows are silently dropped — partial-ingest safe.

In [ ]:
implements_edges = [
    (dk.instrument.id, f'eu/celex/{celex}', celex)
    for celex in dk.cites_eu_celex
]
print('seeds:', implements_edges)

load_implements(implements_edges, store=store)

# Verify what stuck (LED 2016/680 wasn't loaded — that edge dropped).
rows = store.run(
    'MATCH (src:Instrument)-[:IMPLEMENTS]->(tgt:Instrument) '
    'RETURN src.id AS src, tgt.id AS tgt'
)
for r in rows:
    print(f"  {r['src']:<32} IMPLEMENTS {r['tgt']}")

## Step 5 — chunk + embed Provisions

Local sentence-transformers backend (`Qwen/Qwen3-Embedding-0.6B`, 1024-d, CPU-friendly, ~1.2 GB). First run downloads + caches under `~/.cache/huggingface/`; subsequent runs are network-free.

**Vector-index dim must match the embedder dim** — we rebuild the index to the embedder's reported dim if they disagree.

In [ ]:
from crimellm.clg.embed.chunker import chunk_provision
from crimellm.clg.embed.embedder import get_embedder
from crimellm.clg.graph.loaders import vector_index_dim
from crimellm.clg.graph.schema import rebuild_vector_index

# Local sentence-transformers. Swap model via alias: 'qwen' (8B/4096-d),
# 'bge-m3' (1024-d), 'minilm' (384-d). Pass device='mps'/'cuda' for GPU.
embedder = get_embedder('sentence-transformers', model='Qwen/Qwen3-Embedding-0.6B')
print(f'embedder: {embedder.name} (dim={embedder.dim})')

current = vector_index_dim(store)
if current != embedder.dim:
    print(f'rebuilding vector index from {current} → {embedder.dim}')
    rebuild_vector_index(embedder.dim, drop_chunks=True, store=store)

all_provisions = (
    list(eu_reg.provisions) + list(eu_dir.provisions) + list(dk.provisions)
)
chunks = [c for p in all_provisions for c in chunk_provision(p)]
print(f'chunks to embed: {len(chunks)}')

vecs = embedder.embed_batch([c.text for c in chunks])
for c, v in zip(chunks, vecs, strict=True):
    c.embedding = v

n = load_chunks(chunks, embedding_model=embedder.name, store=store)
print(f'loaded {n} chunks')

## Step 6 — verify the vector index resolves up to entity

Pick a chunk, embed its own text, search → top hit should be the same Provision (cosine ≈ 1.0 for a real encoder on identical text).

In [ ]:
from crimellm.clg.graph.loaders import search_chunks

seed = chunks[2]   # pick any chunk
qvec = embedder.embed(seed.text)
hits = search_chunks(qvec, k=3, store=store)
for h in hits:
    print(
        f"{h['score']:.3f} {h['parent_jurisdiction']:<3} "
        f"{h['parent_type']:<10} {h['parent_id']}"
    )

## Summary

In [ ]:
rows = store.run(
    'MATCH (n) WITH labels(n) AS lbls, count(*) AS n '
    'UNWIND lbls AS l RETURN l AS label, n ORDER BY label'
)
for r in rows:
    print(f"  {r['label']:<14} {r['n']:>6}")

## Next: notebook 04 — synthesize a grounded answer

[`04_synthesize_query.ipynb`](04_synthesize_query.ipynb) runs end-to-end queries: EN + DA prompts, jurisdiction inference, cross-jurisdiction IMPLEMENTS traversal, audit-friendly answer metadata.